### Connect postgresql database

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text

# 数据库配置
username = "XXXXXX"
password = "YYYYYY"
host = "localhost"
port = 5432
database = "eyewear-data"

# 创建连接
engine = create_engine(
    f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}"
)

# 查询数据
sql = """
SELECT
    o.order_id,
    o.customer_id,
    o.order_date,
    o.order_status,
    o.total_price_before_tax,
    oi.product_id,
    oi.quantity,
    oi.product_name,
    oi.unit_price,
    oi.line_price_before_tax,
    pi.category,
    pi.sub_category,
    pi.cost_price
FROM "Order" o
JOIN "OrderItem" oi
    ON o.order_id = oi.order_id
JOIN "ProductInfo" pi
    ON oi.product_id = pi.product_id
WHERE o.order_status IN ('Completed', 'Shipped');
"""

df_order_completed_shipped = pd.read_sql(sql, engine)

# 查看数据
df_order_completed_shipped

,order_id,customer_id,order_date,order_status,total_price_before_tax,product_id,quantity,product_name,unit_price,line_price_before_tax,category,sub_category,cost_price
0,733,131843,2023-03-09 02:27:32,Completed,220.67,179,1,Essilor Evolve Progressive,220.67,220.67,Lens,Evolve Lens,70.04
1,734,40409,2023-01-13 21:52:48,Completed,540.53,33,1,Ray-Ban New Wayfarer Prescription,389.25,311.40,Eyeglasses,Eyeglasses,146.67
2,734,40409,2023-01-13 21:52:48,Completed,540.53,71,1,Ray-Ban Kai Prescription,286.41,229.13,Sunglasses,Classic Sunglasses,137.26
3,737,78290,2023-01-03 01:17:35,Completed,246.76,157,1,Ray-Ban Round Metal Prescription,246.76,246.76,Sunglasses,Classic Sunglasses,130.99
4,741,57441,2023-01-14 21:06:22,Completed,167.18,126,1,Ray-Ban Bill Prescription,167.18,167.18,Eyeglasses,Eyeglasses,82.75
...,...,...,...,...,...,...,...,...,...,...,...,...,...
655896,722,105081,2023-01-18 06:38:28,Shipped,439.08,39,1,Ray-Ban Flacko Prescription,284.59,284.59,Sunglasses,Classic Sunglasses,147.17
655897,723,113075,2023-03-18 07:07:55,Completed,295.09,100,1,Ray-Ban Ray-Ban Reverse Prescription,295.09,295.09,AI Glasses,AI Smart Glasses,126.35
655898,724,111989,2023-03-18 22:42:40,Shipped,928.94,41,1,Ray-Ban Original Wayfarer Non-prescription,403.34,403.34,Eyeglasses,Eyeglasses,175.41
655899,724,111989,2023-03-18 22:42:40,Shipped,928.94,188,1,Essilor Chromance Progressive,277.30,277.30,Lens,Chromance Lens,94.27


### Calculate cost per month 

In [ ]:
# 确保 order_date 是 datetime
df_order_completed_shipped["order_date"] = pd.to_datetime(df_order_completed_shipped["order_date"])

# 增加月份列
df_order_completed_shipped["order_quarter"] = df_order_completed_shipped["order_date"].dt.to_period("Q").astype(str).str.replace("Q", "-Q", regex=False)


### Calculate `total_price_before_tax` per order 

In [ ]:
# Order表事先已计算好了`total_price_before_tax`了

### Calculate `gross profit` and `gross margin` per month 

In [6]:
df_revenue_per_order = (
    df_order_completed_shipped[["order_id", "order_quarter", "total_price_before_tax"]]
    .drop_duplicates(subset=["order_id"])
)

df_quarterly = (
    df_revenue_per_order
    .groupby("order_quarter", as_index=False)
    .agg(
        total_revenue=("total_price_before_tax", "sum"),
        order_count=("order_id", "nunique")
    )
)

df_quarterly["avg_order_value"] = df_quarterly["total_revenue"] / df_quarterly["order_count"]

df_quarterly

,order_quarter,total_revenue,order_count,avg_order_value
0,2023-Q1,17289597.00,43954,393.356623
1,2023-Q2,17831533.26,44930,396.873654
2,2023-Q3,25312483.09,62825,402.904625
3,2023-Q4,35905571.61,90014,398.888746
4,2024-Q1,14498928.25,35992,402.837526
5,2024-Q2,22323661.97,53977,413.577301
6,2024-Q3,18840455.54,45049,418.221393
7,2024-Q4,27157147.73,67562,401.958908


In [7]:
# 关闭数据库连接
engine.dispose()